In [4]:
import os, sys
from pathlib import Path

p = Path.cwd()
while not (p / "src").is_dir() and p != p.parent:
    p = p.parent
os.chdir(p)
sys.path.insert(0, str(p / "src"))
print("repo root:", os.getcwd())

repo root: C:\Users\sudar\Desktop\Preparation\tier-a-prep\precrash-eval


In [7]:
import subprocess, sys

subprocess.run([sys.executable, "scripts/make_synthetic_features.py",
                "--manifest", "data/nexar_fixed.dev.csv",
                "--out_dir", "features/synth", "--limit", "120"])

r = subprocess.run([sys.executable, "scripts/protocol_comparison.py",
                    "--features", "features/synth",
                    "--config", "configs/honest.yaml",
                    "--out", "results/synth.json",
                    "--dataset_name", "SYNTHETIC"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr[-500:])

SYNTHETIC: 65 positive, 55 negative clips, per-clip rates 29.10-31.00, median 30.00

  decision windows: positives end at their onset; negatives matched to 135-135 frames (unmatched would be 150)

method                     blind     t_c    TTA*   STTA     | AP     AUC  TTA@R80  rank L->C
--------------------------------------------------------------------------------------------
prior_constant_0.51          yes     0.0    4.98   1.00   0.5417  0.5000      nan     1 -> 11 
prior_sigmoid_m0.40          yes    60.0    2.99   1.00   0.5417  0.5000     0.36     2 -> 5  
prior_sigmoid_m0.50          yes    75.0    2.49   1.00   0.5417  0.5000     0.20     3 -> 6  
prior_linear_ramp            yes    75.0    2.49   1.00   0.5417  0.5000     0.03     4 -> 9  
prior_step_at_half           yes    75.0    2.49   1.00   0.5417  0.5000     1.99     5 -> 10 
prior_sigmoid_m0.60          yes    90.0    1.99   1.00   0.5417  0.5000     0.03     6 -> 7  
prior_sigmoid_m0.70          yes   105.0    1.4

In [8]:
try:
    import torch
    print(torch.__version__, "| cuda:", torch.cuda.is_available())
except ImportError:
    print("no torch installed")

no torch installed


In [9]:
subprocess.run([sys.executable, "scripts/analyse_taa.py",
                "--test_csv", "data/test.csv",
                "--submission_csv", "data/sample_submission.csv",
                "--train_csv", "data/train.csv",
                "--reported_crossover", "22.7",
                "--out", "results/taa_analysis.json"])

CompletedProcess(args=['C:\\Users\\sudar\\Desktop\\Preparation\\tier-a-prep\\precrash-eval\\.venv\\Scripts\\python.exe', 'scripts/analyse_taa.py', '--test_csv', 'data/test.csv', '--submission_csv', 'data/sample_submission.csv', '--train_csv', 'data/train.csv', '--reported_crossover', '22.7', '--out', 'results/taa_analysis.json'], returncode=1)

In [12]:
import subprocess, sys, os
from pathlib import Path

for f in ("test.csv", "sample_submission.csv", "train.csv"):
    p = Path("data") / f
    print(f"{'OK ' if p.exists() else 'MISSING'}  {p}")

r = subprocess.run([sys.executable, "scripts/analyse_taa.py",
                    "--test_csv", "data/test.csv",
                    "--submission_csv", "data/sample_submission.csv",
                    "--train_csv", "data/train.csv",
                    "--reported_crossover", "22.7",
                    "--out", "results/taa_analysis.json"],
                   capture_output=True, text=True)
print(r.stdout)
print("--- stderr ---", r.stderr[-1200:])

OK   data\test.csv
OK   data\sample_submission.csv
OK   data\train.csv
data/test.csv: 1417 clips
columns: ['id', 'video_id', 'start_frame', 'end_frame', 'caption']

WHAT THE CORPUS DOES NOT PUBLISH
  label        ABSENT                 -> average precision, area under the ROC curve, false-positive rate
  event_time   ABSENT                 -> time-to-accident measured to the collision
  alert_time   ABSENT                 -> a ceiling to check a reported warning time against

  train.csv: 1 row(s)
    note: This is a zero-shot traffic accident anticipation task. There is no train split. Please use test_kaggle.csv as the only public data for inference.

  window spans: {150: 1417}

THE ORGANISERS' REFERENCE SUBMISSION
  entries              : 1417
  length               : 150
  first, last          : 0.001000, 0.999000
  identical every clip : True
  max deviation from a straight line: 5.10e-07

  One curve, repeated for every clip. It reads no pixels.

CROSSING FRAME OF A VIDEO-BLIND F

In [11]:
import os, subprocess, sys
from getpass import getpass

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"])
os.environ["KAGGLE_API_TOKEN"] = getpass("Kaggle token (KGAT_...): ").strip()

os.makedirs("data", exist_ok=True)
for name in ("test.csv", "sample_submission.csv", "train.csv"):
    r = subprocess.run([sys.executable, "-m", "kaggle", "competitions", "download",
                        "-c", "zero-shot-taa", "-f", name, "-p", "data"],
                       capture_output=True, text=True)
    print(name, "->", r.returncode, r.stderr[-200:] if r.returncode else "")

import zipfile, glob
for z in glob.glob("data/*.zip"):
    with zipfile.ZipFile(z) as f: f.extractall("data")
    os.remove(z)

print(sorted(os.listdir("data")))

Kaggle token (KGAT_...):  ········


test.csv -> 0 
sample_submission.csv -> 0 
train.csv -> 0 
['nexar_fixed.csv', 'nexar_fixed.dev.csv', 'nexar_fixed.test.csv', 'nexar_random.csv', 'nexar_random.dev.csv', 'nexar_random.test.csv', 'sample_submission.csv', 'test.csv', 'train.csv']
